# Lesson 1 demo: tokens, cost, thinking, sampling and three prompt patterns

Run it from top to bottom. Every cell's output below is what it printed when
the teacher ran it; **yours will differ**, because a model does not give the
same answer twice, and models are updated. Compare the shapes, not the digits.

**Before you start:** your key is in a `.env` file next to this notebook
(`ANTHROPIC_API_KEY=...`), and the packages in `requirements.txt` are
installed. See the lesson's setup page.

In [1]:
import csv
import datetime
import json
import re
import urllib.request
from collections import Counter
from pathlib import Path

import anthropic
import tiktoken
from dotenv import load_dotenv

load_dotenv()                  # reads ANTHROPIC_API_KEY from .env
client = anthropic.Anthropic() # never write the key in the code

HAIKU = "claude-haiku-4-5"     # the model you use in this course
OPUS = "claude-opus-5-5"       # used once, for comparison
PRICE = {HAIKU: (1.00, 5.00), OPUS: (4.00, 20.00)}  # $ per million tokens, in / out (Anthropic, 1 Oct 2026)

print("run on", datetime.date.today(), "| anthropic", anthropic.__version__, "| tiktoken", tiktoken.__version__)

run on 2026-10-01 | anthropic 1.10.0 | tiktoken 0.12.0


In [2]:
# The 120 labelled support tickets, from the company's public release
URL = ("https://github.com/TangoAI-AlbertSchool-Data-Eng-Cloud/albert-marketplace"
       "/releases/download/tickets-v1.0.0/tickets_v1.csv")
path = Path("data/tickets_v1.csv")
if not path.exists():
    path.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(URL, path)

tickets = {row["ticket_id"]: row for row in csv.DictReader(path.open(encoding="utf-8"))}
print(len(tickets), "tickets")
print(tickets["T020"]["subject"])

120 tickets
Order 181571 still hasn't shown up


## 1. The SDK in one call

Everything the Anthropic SDK does rests on one idea: **`client.messages.create(...)`
sends one request and returns one `Message`.** Everything else is a parameter of
the request or a field of the response.

In [3]:
response = client.messages.create(
    model=HAIKU,
    max_tokens=50,
    messages=[{"role": "user", "content": "Say hello to a new student in five words."}],
)

print(type(response).__name__)
for block in response.content:          # content is a LIST of blocks, not a string
    print(block.type, "->", block.text)
print("stop_reason:", response.stop_reason)
print("usage:", response.usage.input_tokens, "input tokens,", response.usage.output_tokens, "output tokens")

Message
text -> Hello, welcome to our class!
stop_reason: end_turn
usage: 17 input tokens, 10 output tokens


In [4]:
# The context window, asked of the API itself rather than remembered
for model in [HAIKU, OPUS]:
    info = client.models.retrieve(model)
    print(f"{model}: {info.max_input_tokens:,} input tokens, {info.max_tokens:,} output tokens")

claude-haiku-4-5: 200,000 input tokens, 64,000 output tokens


claude-opus-5-5: 1,000,000 input tokens, 128,000 output tokens


## 2. Tokens, and what they cost

An open tokenizer shows **how** text is cut. Claude's own tokenizer is not
published, so the **count** that is billed comes from Anthropic's
token-counting endpoint, which is free.

In [5]:
enc = tiktoken.get_encoding("o200k_base")
sentence = "Hi, I ordered the Sally Hansen Miracle Gel Set, order number 181571. Could you check where my parcel is?"
ids = enc.encode(sentence)
print(len(sentence), "characters ->", len(ids), "tokens")
print([enc.decode([i]) for i in ids])
print(ids)

104 characters -> 25 tokens
['Hi', ',', ' I', ' ordered', ' the', ' Sally', ' Hansen', ' Miracle', ' Gel', ' Set', ',', ' order', ' number', ' ', '181', '571', '.', ' Could', ' you', ' check', ' where', ' my', ' parcel', ' is', '?']
[12194, 11, 357, 17889, 290, 88213, 88786, 120276, 28986, 3957, 11, 2569, 2086, 220, 16813, 41467, 13, 30456, 481, 2371, 1919, 922, 37708, 382, 30]


In [6]:
# Ticket T020 in English, and translated into French and German (test material)
texts = json.loads(Path("data/t020_translations.json").read_text(encoding="utf-8"))
english = None
for lang, text in texts.items():
    n = client.messages.count_tokens(model=HAIKU, messages=[{"role": "user", "content": text}]).input_tokens
    english = english or n
    print(f"{lang}: {len(text.split()):3} words, {n} tokens ({n / english - 1:+.0%}), "
          f"${n * 10_000 / 1e6 * PRICE[HAIKU][0]:.2f} per 10,000 messages")

en: 103 words, 161 tokens (+0%), $1.61 per 10,000 messages


fr: 110 words, 234 tokens (+45%), $2.34 per 10,000 messages


de: 103 words, 229 tokens (+42%), $2.29 per 10,000 messages


## 3. Output tokens: one label, two models

The same instruction, *answer with the label only*, on the ten tickets of the
test set. Every visible answer is one word; watch `usage.output_tokens`.

In [7]:
LABELS = ["billing", "delivery", "returns_refunds", "account", "seller", "product", "information", "other"]
SYSTEM = (
    "You route customer messages for Albert's Marketplace, a European online marketplace for beauty products. Labels:\n"
    "billing: a charge, a payment, a way to pay\n"
    "delivery: where a parcel is, or when it came\n"
    "returns_refunds: sending something back, getting money back\n"
    "account: signing in, the subscription, the account itself\n"
    "seller: what a third-party seller did or sent\n"
    "product: the product itself, its quality, not as expected\n"
    "information: a question about using a product or the service\n"
    "other: anything else"
)
TEST = ["T005", "T008", "T016", "T018", "T023", "T024", "T032", "T034", "T042", "T045"]
SHOTS = ["T002", "T020", "T003", "T014", "T012"]   # outside the test set, five different labels
spent = Counter()                                    # every token this notebook uses


def text_of(t):
    return f"Subject: {t['subject']}\n\n{t['body']}"


def ask(model, user, system=SYSTEM, max_tokens=4000, **extra):
    r = client.messages.create(model=model, max_tokens=max_tokens, system=system,
                               messages=[{"role": "user", "content": user}], **extra)
    spent[model, "in"] += r.usage.input_tokens
    spent[model, "out"] += r.usage.output_tokens
    return r


def answer_text(r):
    return "".join(b.text for b in r.content if b.type == "text")


def parse(text):
    found = re.findall(r"\b(" + "|".join(LABELS) + r")\b", text.lower())
    return found[-1] if found else ""


def cost(model, tokens_in, tokens_out):
    p_in, p_out = PRICE[model]
    return tokens_in / 1e6 * p_in + tokens_out / 1e6 * p_out

In [8]:
for model, extra in [(HAIKU, {}), (OPUS, {"output_config": {"effort": "low"}})]:
    tin = tout = 0
    answers = []
    for tid in TEST:
        r = ask(model, f"{text_of(tickets[tid])}\n\nAnswer with the label only.", **extra)
        tin += r.usage.input_tokens
        tout += r.usage.output_tokens
        answers.append(answer_text(r).strip())
    print(f"{model}: {tout} output tokens for ten answers, ${cost(model, tin, tout):.4f}")
    print("   answers:", answers)

claude-haiku-4-5: 46 output tokens for ten answers, $0.0035
   answers: ['billing', 'returns_refunds', 'delivery', 'billing', 'information', 'account', 'delivery', 'returns_refunds', 'product', 'product']


claude-opus-5-5: 142 output tokens for ten answers, $0.0201
   answers: ['billing', 'returns_refunds', 'delivery', 'billing', 'information', 'account', 'delivery', 'seller', 'returns_refunds', 'product']


## 4. Model thinking: what you can read, what you pay for

Ticket T001: a customer unhappy with a small bottle for its price, who also
asks for their money back. Its label is `product`; `returns_refunds` fits too.

In [9]:
t001 = f"{text_of(tickets['T001'])}\n\nAnswer with the label only."

r = ask(HAIKU, t001, thinking={"type": "enabled", "budget_tokens": 2000})
thinking = "".join(b.thinking for b in r.content if b.type == "thinking")
print("Haiku 4.5, thinking on. Answer:", answer_text(r).strip())
print("output tokens billed:", r.usage.output_tokens)
print("--- the thinking, as returned ---")
print(thinking)

Haiku 4.5, thinking on. Answer: returns_refunds
output tokens billed: 603
--- the thinking, as returned ---
Let me analyze this customer message to determine the correct label.

The customer ordered an Egyptian Musk oil (5ml bottle) and is complaining about:
1. The product size being too small for the price
2. The packaging being excessive
3. Wanting a refund

Let me check each potential label:

- **billing**: This is about a charge/payment. The customer mentions the price (12.38 EUR) but the complaint isn't about the charge itself being incorrect, it's about the product quality/value.

- **delivery**: This is about where a parcel is or when it came. The customer mentions the delivery (arrived 30 April) but the complaint isn't about the delivery itself.

- **returns_refunds**: This is about sending something back and getting money back. The customer explicitly says "I'd like my money back if possible" at the end. They want a refund.

- **account**: Not relevant - no account or subscrip

In [10]:
r = ask(OPUS, t001, output_config={"effort": "medium"})   # Opus 5.5 with its default settings
blocks = [b for b in r.content if b.type == "thinking"]
print("Opus 5.5, default. Answer:", answer_text(r).strip())
print("thinking blocks:", len(blocks), "| characters of thinking text returned:", sum(len(b.thinking) for b in blocks))
print("output tokens billed:", r.usage.output_tokens)

Opus 5.5, default. Answer: returns_refunds
thinking blocks: 1 | characters of thinking text returned: 0
output tokens billed: 94


## 5. Sampling: what the current models still let you set

In version 1 of the SDK, `temperature` is no longer an argument of
`messages.create`; it can only be sent through `extra_body`, and the model
decides whether to accept it.

In [11]:
try:
    ask(OPUS, "Say OK.", max_tokens=16, extra_body={"temperature": 0.0})
    print("accepted")
except anthropic.BadRequestError as e:
    print("Opus 5.5 with temperature 0 ->", e.status_code, e.message)

Opus 5.5 with temperature 0 -> 400 Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': '`temperature` is deprecated for this model.'}, 'request_id': 'req_011CfboUgJFVaDtXpe6jRuXP'}


In [12]:
# The same ambiguous ticket (T018: billing, or account), ten times at each temperature, on Haiku 4.5
t018 = f"{text_of(tickets['T018'])}\n\nAnswer with the label only."
for temperature in [1.0, 0.0]:
    labels = Counter(parse(answer_text(ask(HAIKU, t018, max_tokens=20, extra_body={"temperature": temperature})))
                     for _ in range(10))
    print(f"temperature {temperature}: {dict(labels)}")

temperature 1.0: {'account': 1, 'billing': 9}


temperature 0.0: {'billing': 10}


## 6. Three prompt patterns, measured on the test set

Errors are counted twice: against the ticket's label only (**strict**), and
counting its second acceptable label as right too (**either**).

In [13]:
def shots_block():
    return "\n\n".join(f"Message:\n{text_of(tickets[s])}\nLabel: {tickets[s]['gold_label']}" for s in SHOTS)


PATTERNS = {
    "zero-shot": lambda t: f"{text_of(t)}\n\nAnswer with the label only.",
    "five-shot": lambda t: f"{shots_block()}\n\nMessage:\n{text_of(t)}\n\nAnswer with the label only.\nLabel:",
    "chain-of-thought": lambda t: (f"{text_of(t)}\n\nThink step by step about what the customer needs, "
                                   "then give the label on the last line as 'Label: <label>'."),
}


def measure(model, build, **extra):
    rows, tin, tout = [], 0, 0
    for tid in TEST:
        t = tickets[tid]
        r = ask(model, build(t), **extra)
        tin += r.usage.input_tokens
        tout += r.usage.output_tokens
        rows.append((tid, t["gold_label"], t["ambiguous_with"], parse(answer_text(r))))
    strict = sum(got != gold for _, gold, _, got in rows)
    either = sum(got not in (gold, second) for _, gold, second, got in rows)
    return strict, either, tin, tout, rows


print(f"{'pattern':18} {'strict':>6} {'either':>6} {'in':>7} {'out':>6} {'cost':>8}")
for name, build in PATTERNS.items():
    strict, either, tin, tout, rows = measure(HAIKU, build)
    print(f"{name:18} {strict:>4}/10 {either:>4}/10 {tin:>7} {tout:>6} ${cost(HAIKU, tin, tout):.4f}")
print("model:", HAIKU, "| date:", datetime.date.today(), "| test file: tickets-v1.0.0, 10 tickets")

pattern            strict either      in    out     cost


zero-shot             2/10    2/10    3273     46 $0.0035


five-shot             3/10    3/10   13083     42 $0.0133


chain-of-thought      4/10    3/10    3483   2535 $0.0162
model: claude-haiku-4-5 | date: 2026-10-01 | test file: tickets-v1.0.0, 10 tickets


## 7. One sentence about the format

The same five-shot prompt on Opus 5.5, without and then with
*"Answer with the label only."* Nothing else changes.

In [14]:
no_rule = lambda t: f"{shots_block()}\n\nMessage:\n{text_of(t)}\nLabel:"
for name, build in [("without the rule", no_rule), ("with the rule", PATTERNS["five-shot"])]:
    strict, either, tin, tout, rows = measure(OPUS, build, output_config={"effort": "low"})
    print(f"Opus 5.5, five-shot {name}: {either}/10 errors (either label), {tout} output tokens")

r = ask(OPUS, no_rule(tickets["T042"]), output_config={"effort": "low"})
print("--- one answer without the rule (T042) ---")
print(answer_text(r)[:700])
print("--- what the parser took:", parse(answer_text(r)), "| the right label:", tickets["T042"]["gold_label"])

Opus 5.5, five-shot without the rule: 3/10 errors (either label), 1038 output tokens


Opus 5.5, five-shot with the rule: 2/10 errors (either label), 317 output tokens


--- one answer without the rule (T042) ---
**Label: product**

The core issue is the condition of the item itself: the primer arrived expired. That is a product quality problem, like the flimsy eyelashes in an earlier example. The request for a replacement or credit is secondary. This isn't a seller case like the missing eyeliner creams, where part of the order never arrived.
--- what the parser took: seller | the right label: product


In [15]:
total = sum(cost(m, spent[m, "in"], spent[m, "out"]) for m in PRICE)
for m in PRICE:
    print(f"{m}: {spent[m, 'in']:,} input + {spent[m, 'out']:,} output tokens, ${cost(m, spent[m, 'in'], spent[m, 'out']):.4f}")
print(f"this notebook, run once: ${total:.2f}")

claude-haiku-4-5: 32,525 input + 3,354 output tokens, $0.0493
claude-opus-5-5: 40,168 input + 1,857 output tokens, $0.1978
this notebook, run once: $0.25
